# 03: 特徴量エンジニアリングと学習データ生成

予測表から特徴量を抽出し、AIモデルの学習データを生成します。

## 目的
- 予測表から特徴量を抽出する処理の実装
- パターンIDを特徴量として追加する処理の実装
- 軸数字予測モデルの学習データ生成
- 組み合わせ予測モデルの学習データ生成

## データ構造
- **軸数字予測**: 100回分 × 4パターン × 10数字 = 4,000サンプル
- **組み合わせ予測**: 100回分 × 4パターン × 数百〜数千サンプル = 数万〜数十万サンプル


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
from itertools import product
warnings.filterwarnings('ignore')

# プロジェクトルートのパスを設定
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

# モジュールをインポート
import sys
sys.path.append(str(PROJECT_ROOT / 'notebooks'))
from chart_generator import (
    load_keisen_master,
    generate_chart,
    Pattern,
    Target
)
from feature_extractor import (
    extract_digit_features,
    extract_combination_features,
    add_pattern_id_features,
    features_to_vector,
    get_rehearsal_positions
)

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"データディレクトリ: {DATA_DIR}")


In [ ]:
# データの読み込み
train_csv_path = DATA_DIR / 'train_data_100.csv'

if train_csv_path.exists():
    train_df = pd.read_csv(train_csv_path)
    print(f"学習用データセット: {len(train_df)}回分")
else:
    # なければ過去実績データから読み込む
    csv_path = DATA_DIR / 'past_results.csv'
    df = pd.read_csv(csv_path)
    df = df.sort_values('round_number', ascending=False).reset_index(drop=True)
    
    # 基準回号（6758）からさかのぼって100回分を取得
    BASE_ROUND = 6758
    base_idx = df[df['round_number'] == BASE_ROUND].index[0]
    train_df = df.iloc[base_idx:base_idx+100].copy()
    train_df = train_df.sort_values('round_number', ascending=True).reset_index(drop=True)
    print(f"学習用データセット: {len(train_df)}回分（直接読み込み）")

# 当選番号を文字列型に変換（数値型で読み込まれる可能性があるため）
train_df['n3_winning'] = train_df['n3_winning'].astype(str).str.replace('.0', '', regex=False)
train_df['n4_winning'] = train_df['n4_winning'].astype(str).str.replace('.0', '', regex=False)

print(f"回号範囲: {train_df['round_number'].min()} 〜 {train_df['round_number'].max()}")

# 罫線マスターデータの読み込み
keisen_master = load_keisen_master(DATA_DIR)
print("\n罫線マスターデータの読み込み完了")


In [ ]:
# テスト: 特徴量抽出の動作確認
test_round = 6758
test_target: Target = 'n3'
test_pattern: Pattern = 'A1'

# 予測表を生成
grid, rows, cols = generate_chart(
    train_df, keisen_master, test_round, test_pattern, test_target
)

# リハーサル数字を取得（前回の当選番号）
rehearsal_row = train_df[train_df['round_number'] == test_round - 1]
if len(rehearsal_row) > 0:
    rehearsal_digits = rehearsal_row['n3_winning'].iloc[0] if test_target == 'n3' else rehearsal_row['n4_winning'].iloc[0]
    rehearsal_positions = get_rehearsal_positions(grid, rows, cols, rehearsal_digits)
else:
    rehearsal_positions = None

# テスト数字（例: 5）の特徴量を抽出
test_digit = 5
features = extract_digit_features(
    grid, rows, cols, test_digit, rehearsal_positions
)
features = add_pattern_id_features(features, test_pattern)

print(f"テスト回号: {test_round}, パターン: {test_pattern}, 対象: {test_target}")
print(f"テスト数字: {test_digit}")
print(f"\n抽出された特徴量数: {len(features)}")
print(f"\n特徴量の例（最初の10個）:")
for i, (key, value) in enumerate(list(features.items())[:10]):
    print(f"  {key}: {value}")


## 軸数字予測モデルの学習データ生成

各回号×4パターン×各数字（0-9）のサンプルを生成します。


In [ ]:
# 軸数字予測モデルの学習データ生成
patterns: List[Pattern] = ['A1', 'A2', 'B1', 'B2']
targets: List[Target] = ['n3', 'n4']

axis_samples = []

print("軸数字予測モデルの学習データを生成中...")

for _, row in train_df.iterrows():
    round_number = row['round_number']
    
    # リハーサル数字を取得（前回の当選番号）
    rehearsal_row = train_df[train_df['round_number'] == round_number - 1]
    
    for target in targets:
        # リハーサル数字を取得
        if len(rehearsal_row) > 0:
            rehearsal_digits_raw = rehearsal_row[f'{target}_winning'].iloc[0]
            # 文字列型に変換（数値型の場合に備えて）
            rehearsal_digits = str(rehearsal_digits_raw).replace('.0', '')
        else:
            rehearsal_digits = None
        
        for pattern in patterns:
            try:
                # 予測表を生成
                grid, rows, cols = generate_chart(
                    train_df, keisen_master, round_number, pattern, target
                )
                
                # リハーサル位置を取得
                if rehearsal_digits:
                    rehearsal_positions = get_rehearsal_positions(
                        grid, rows, cols, rehearsal_digits
                    )
                else:
                    rehearsal_positions = None
                
                # 各数字（0-9）のサンプルを生成
                for digit in range(10):
                    # 特徴量を抽出
                    features = extract_digit_features(
                        grid, rows, cols, digit, rehearsal_positions
                    )
                    
                    # パターンIDを追加
                    features = add_pattern_id_features(features, pattern)
                    
                    # ラベルを生成（当選番号に含まれたか）
                    winning = str(row[f'{target}_winning']).replace('.0', '')
                    label = 1 if str(digit) in winning else 0
                    
                    axis_samples.append({
                        'round_number': round_number,
                        'target': target,
                        'pattern': pattern,
                        'digit': digit,
                        'features': features,
                        'label': label
                    })
                    
            except Exception as e:
                print(f"エラー: 回号={round_number}, 対象={target}, パターン={pattern}: {e}")

print(f"\n生成されたサンプル数: {len(axis_samples)}")
print(f"期待値: {len(train_df)}回 × {len(targets)}対象 × {len(patterns)}パターン × 10数字 = {len(train_df) * len(targets) * len(patterns) * 10}")

# ラベルの分布を確認
labels = [s['label'] for s in axis_samples]
print(f"\nラベルの分布:")
print(f"  0（含まれない）: {labels.count(0)} ({labels.count(0)/len(labels)*100:.1f}%)")
print(f"  1（含まれる）: {labels.count(1)} ({labels.count(1)/len(labels)*100:.1f}%)")


In [ ]:
# 軸数字予測モデルの学習データをベクトル化
print("特徴量ベクトルに変換中...")

# 特徴量のキーを取得（最初のサンプルから）
feature_keys = sorted(axis_samples[0]['features'].keys())
print(f"特徴量の次元数: {len(feature_keys)}")

# 特徴量とラベルをベクトル化
X_axis = []
y_axis = []
metadata_axis = []  # 回号、対象、パターン、数字を保存

for sample in axis_samples:
    feature_vector = features_to_vector(sample['features'])
    X_axis.append(feature_vector)
    y_axis.append(sample['label'])
    metadata_axis.append({
        'round_number': sample['round_number'],
        'target': sample['target'],
        'pattern': sample['pattern'],
        'digit': sample['digit']
    })

X_axis = np.array(X_axis)
y_axis = np.array(y_axis)

print(f"\n特徴量行列の形状: {X_axis.shape}")
print(f"ラベルベクトルの形状: {y_axis.shape}")
print(f"\n特徴量の統計:")
print(f"  平均: {X_axis.mean(axis=0)[:5]}...")  # 最初の5次元のみ表示
print(f"  標準偏差: {X_axis.std(axis=0)[:5]}...")


## 組み合わせ予測モデルの学習データ生成

各回号×4パターン×全組み合わせ（またはサンプリング）のサンプルを生成します。


In [ ]:
# 組み合わせ予測モデルの学習データ生成
# 注意: 全組み合わせを生成するとデータ量が膨大になるため、
# サンプリング戦略を採用します（例: 各数字から上位5組み合わせを抽出）

MAX_COMBINATIONS_PER_DIGIT = 20  # 各数字ごとの最大組み合わせ数
comb_samples = []

print("組み合わせ予測モデルの学習データを生成中...")

for _, row in train_df.iterrows():
    round_number = row['round_number']
    
    # リハーサル数字を取得（前回の当選番号）
    rehearsal_row = train_df[train_df['round_number'] == round_number - 1]
    
    for target in targets:
        # リハーサル数字を取得
        if len(rehearsal_row) > 0:
            rehearsal_digits = rehearsal_row[f'{target}_winning'].iloc[0]
        else:
            rehearsal_digits = None
        
        # 当選番号を取得
        winning = row[f'{target}_winning']
        
        for pattern in patterns:
            try:
                # 予測表を生成
                grid, rows, cols = generate_chart(
                    train_df, keisen_master, round_number, pattern, target
                )
                
                # リハーサル位置を取得
                if rehearsal_digits:
                    rehearsal_positions = get_rehearsal_positions(
                        grid, rows, cols, rehearsal_digits
                    )
                else:
                    rehearsal_positions = None
                
                # 組み合わせを生成
                # N3の場合: 3桁の組み合わせ（000-999）
                # N4の場合: 4桁の組み合わせ（0000-9999）
                digit_count = 3 if target == 'n3' else 4
                
                # 全組み合わせを生成すると多すぎるため、サンプリング
                # 簡易版: 各数字（0-9）を含む組み合わせを一定数生成
                combinations_generated = set()
                
                # 各数字を含む組み合わせを生成
                for digit in range(10):
                    # その数字を含む組み合わせを生成
                    for _ in range(MAX_COMBINATIONS_PER_DIGIT):
                        # ランダムに組み合わせを生成（簡易版）
                        # 実際の実装では、予測表から候補を抽出する方が良い
                        if target == 'n3':
                            # N3: 3桁の組み合わせ（digitを含む）
                            other_digits = [d for d in range(10) if d != digit]
                            if len(other_digits) >= 2:
                                other_two = np.random.choice(other_digits, size=2, replace=False)
                                combo = ''.join(map(str, sorted([digit] + list(other_two))))
                            else:
                                combo = str(digit) * 3
                        else:
                            # N4: 4桁の組み合わせ（digitを含む）
                            other_digits = [d for d in range(10) if d != digit]
                            if len(other_digits) >= 3:
                                other_three = np.random.choice(other_digits, size=3, replace=False)
                                combo = ''.join(map(str, sorted([digit] + list(other_three))))
                            else:
                                combo = str(digit) * 4
                        
                        if combo not in combinations_generated:
                            combinations_generated.add(combo)
                            
                            # 特徴量を抽出
                            features = extract_combination_features(
                                grid, rows, cols, combo, rehearsal_positions
                            )
                            
                            # パターンIDを追加
                            features = add_pattern_id_features(features, pattern)
                            
                            # ラベルを生成（当選したか）
                            # ボックスの場合: 順序無関係、ストレートの場合: 順序重要
                            label_box = 1 if set(combo) == set(winning) else 0
                            label_straight = 1 if combo == winning else 0
                            
                            comb_samples.append({
                                'round_number': round_number,
                                'target': target,
                                'pattern': pattern,
                                'combination': combo,
                                'features': features,
                                'label_box': label_box,
                                'label_straight': label_straight
                            })
                            
                            if len(combinations_generated) >= MAX_COMBINATIONS_PER_DIGIT * 10:
                                break
                    
                    if len(combinations_generated) >= MAX_COMBINATIONS_PER_DIGIT * 10:
                        break
                    
            except Exception as e:
                print(f"エラー: 回号={round_number}, 対象={target}, パターン={pattern}: {e}")

print(f"\n生成されたサンプル数: {len(comb_samples)}")

# ラベルの分布を確認
labels_box = [s['label_box'] for s in comb_samples]
labels_straight = [s['label_straight'] for s in comb_samples]
print(f"\nラベルの分布（ボックス）:")
print(f"  0（当選なし）: {labels_box.count(0)} ({labels_box.count(0)/len(labels_box)*100:.1f}%)")
print(f"  1（当選あり）: {labels_box.count(1)} ({labels_box.count(1)/len(labels_box)*100:.1f}%)")
print(f"\nラベルの分布（ストレート）:")
print(f"  0（当選なし）: {labels_straight.count(0)} ({labels_straight.count(0)/len(labels_straight)*100:.1f}%)")
print(f"  1（当選あり）: {labels_straight.count(1)} ({labels_straight.count(1)/len(labels_straight)*100:.1f}%)")


In [ ]:
# 組み合わせ予測モデルの学習データをベクトル化
print("特徴量ベクトルに変換中...")

# 特徴量のキーを取得（最初のサンプルから）
comb_feature_keys = sorted(comb_samples[0]['features'].keys())
print(f"特徴量の次元数: {len(comb_feature_keys)}")

# 特徴量とラベルをベクトル化
X_comb = []
y_comb_box = []
y_comb_straight = []
metadata_comb = []  # 回号、対象、パターン、組み合わせを保存

for sample in comb_samples:
    feature_vector = features_to_vector(sample['features'])
    X_comb.append(feature_vector)
    y_comb_box.append(sample['label_box'])
    y_comb_straight.append(sample['label_straight'])
    metadata_comb.append({
        'round_number': sample['round_number'],
        'target': sample['target'],
        'pattern': sample['pattern'],
        'combination': sample['combination']
    })

X_comb = np.array(X_comb)
y_comb_box = np.array(y_comb_box)
y_comb_straight = np.array(y_comb_straight)

print(f"\n特徴量行列の形状: {X_comb.shape}")
print(f"ラベルベクトルの形状（ボックス）: {y_comb_box.shape}")
print(f"ラベルベクトルの形状（ストレート）: {y_comb_straight.shape}")


## データセットの分割と保存

学習データと検証データに分割し、保存します。


In [ ]:
from sklearn.model_selection import train_test_split

# 軸数字予測データの分割
# 回号ベースで分割（時系列データのため）
unique_rounds = sorted(set([m['round_number'] for m in metadata_axis]))
train_rounds = unique_rounds[:int(len(unique_rounds) * 0.8)]
val_rounds = unique_rounds[int(len(unique_rounds) * 0.8):]

train_indices_axis = [i for i, m in enumerate(metadata_axis) if m['round_number'] in train_rounds]
val_indices_axis = [i for i, m in enumerate(metadata_axis) if m['round_number'] in val_rounds]

X_axis_train = X_axis[train_indices_axis]
X_axis_val = X_axis[val_indices_axis]
y_axis_train = y_axis[train_indices_axis]
y_axis_val = y_axis[val_indices_axis]

print(f"軸数字予測データ:")
print(f"  学習データ: {X_axis_train.shape[0]}サンプル")
print(f"  検証データ: {X_axis_val.shape[0]}サンプル")

# 組み合わせ予測データの分割
train_indices_comb = [i for i, m in enumerate(metadata_comb) if m['round_number'] in train_rounds]
val_indices_comb = [i for i, m in enumerate(metadata_comb) if m['round_number'] in val_rounds]

X_comb_train = X_comb[train_indices_comb]
X_comb_val = X_comb[val_indices_comb]
y_comb_box_train = y_comb_box[train_indices_comb]
y_comb_box_val = y_comb_box[val_indices_comb]
y_comb_straight_train = y_comb_straight[train_indices_comb]
y_comb_straight_val = y_comb_straight[val_indices_comb]

print(f"\n組み合わせ予測データ:")
print(f"  学習データ: {X_comb_train.shape[0]}サンプル")
print(f"  検証データ: {X_comb_val.shape[0]}サンプル")


In [ ]:
# データセットの保存
import pickle

# 保存ディレクトリを作成
MODELS_DIR = DATA_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# 軸数字予測データの保存（N3/N4で分ける）
for target in ['n3', 'n4']:
    target_indices_train = [i for i, m in enumerate(metadata_axis) 
                           if m['round_number'] in train_rounds and m['target'] == target]
    target_indices_val = [i for i, m in enumerate(metadata_axis) 
                         if m['round_number'] in val_rounds and m['target'] == target]
    
    if len(target_indices_train) > 0:
        data_file = MODELS_DIR / f'{target}_axis_data.pkl'
        with open(data_file, 'wb') as f:
            pickle.dump({
                'X_train': X_axis[target_indices_train],
                'X_val': X_axis[target_indices_val],
                'y_train': y_axis[target_indices_train],
                'y_val': y_axis[target_indices_val],
                'feature_keys': feature_keys,
                'metadata_train': [metadata_axis[i] for i in target_indices_train],
                'metadata_val': [metadata_axis[i] for i in target_indices_val]
            }, f)
        print(f"保存完了: {data_file}")

# 組み合わせ予測データの保存（N3/N4 × ボックス/ストレートで分ける）
for target in ['n3', 'n4']:
    for combo_type in ['box', 'straight']:
        target_indices_train = [i for i, m in enumerate(metadata_comb) 
                               if m['round_number'] in train_rounds and m['target'] == target]
        target_indices_val = [i for i, m in enumerate(metadata_comb) 
                             if m['round_number'] in val_rounds and m['target'] == target]
        
        if len(target_indices_train) > 0:
            y_train = y_comb_box[target_indices_train] if combo_type == 'box' else y_comb_straight[target_indices_train]
            y_val = y_comb_box[target_indices_val] if combo_type == 'box' else y_comb_straight[target_indices_val]
            
            data_file = MODELS_DIR / f'{target}_{combo_type}_comb_data.pkl'
            with open(data_file, 'wb') as f:
                pickle.dump({
                    'X_train': X_comb[target_indices_train],
                    'X_val': X_comb[target_indices_val],
                    'y_train': y_train,
                    'y_val': y_val,
                    'feature_keys': comb_feature_keys,
                    'metadata_train': [metadata_comb[i] for i in target_indices_train],
                    'metadata_val': [metadata_comb[i] for i in target_indices_val]
                }, f)
            print(f"保存完了: {data_file}")

print(f"\nすべてのデータセットを保存しました: {MODELS_DIR}")


## 次のステップ

特徴量エンジニアリングと学習データ生成が完了しました。次のNotebook (`04_model_training.ipynb`) で、XGBoostモデルを学習します。
